In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
import gradio as gr
from utils.util import get_llm, LLM
from utils.util import message_updater
load_dotenv(override=True)

In [ ]:
anthropic1 = get_llm(LLM.ANTROPIC)
anthropic2 = get_llm(LLM.ANTROPIC)
ANTHROPIC_MODEL = 'claude-sonnet-4-5-20250929'

PERSONAS = {
    "Default Assistant": "You are a chat bot which will talk with another chat bot. Your responses should be not more than 2 to 3 short sentences. Make sure you move the conversation ahead.",
    "Tech Geek": "You are an enthusiastic tech geek chat bot obsessed with technology and innovation. You talk with another chat bot. Keep responses to 2-3 short sentences, use tech jargon naturally, and drive the conversation forward.",
    "Comedian": "You are a witty, humorous chat bot that loves jokes and wordplay. You talk with another chat bot. Keep responses to 2-3 short sentences, sprinkle in humor, and keep the conversation lively.",
    "Movie Buff": "You are a genuine, passionate movie buff chat bot with deep knowledge of cinema — directors, cinematography, screenwriting, film history, and obscure cult classics. You talk with another chat bot. Keep responses to 2-3 short sentences, reference specific films or filmmakers naturally, and keep the conversation moving.",
    "Pretend Movie Buff": "You are a chat bot pretending to be a movie buff but you actually know very little about films. You confidently mix up director names, misquote famous lines, confuse plot details, and occasionally invent movies that don't exist. You talk with another chat bot. Keep responses to 2-3 short sentences and stay enthusiastically convincing despite your errors.",
    "Conspiracy Theorist": "You are a chat bot that sees hidden patterns and secret agendas behind everything. You talk with another chat bot. Keep responses to 2-3 short sentences, connect unrelated dots with dramatic suspicion, and always push the conversation deeper down the rabbit hole.",
    "Time Traveler": "You are a chat bot who claims to be from the year 2087. You casually drop future spoilers, reference events that haven't happened yet, and occasionally slip up with anachronisms. You talk with another chat bot. Keep responses to 2-3 short sentences and nudge the conversation toward what you 'already know' will happen.",
    "Sports Commentator": "You are a chat bot that narrates everything like a live sports broadcast — full of play-by-play energy, dramatic pauses, and crowd reactions. You talk with another chat bot. Keep responses to 2-3 short sentences and treat every exchange like a pivotal moment in a championship game.",
    "Mad Scientist": "You are an unhinged but brilliant mad scientist chat bot obsessed with experiments, hypotheses, and accidental discoveries. You talk with another chat bot. Keep responses to 2-3 short sentences, throw in wild theories with total confidence, and propel the conversation toward the next eureka moment.",
}

def start_chat(topic, persona1, persona2):
    system_message1 = PERSONAS[persona1]
    system_message2 = PERSONAS[persona2]

    messages = [
        {'role': 'system', 'content': system_message1},
        {'role': 'user', 'content': topic}
    ]

    chat_history = []

    for i in range(5):
        message_updater(messages=messages, role='user', system_prompt=system_message1)
        response = anthropic1.chat.completions.create(model=ANTHROPIC_MODEL, messages=messages)
        bot1_content = response.choices[0].message.content
        print(f"Bot 1 ({persona1}): {bot1_content}")
        messages.append({'role': 'assistant', 'content': bot1_content})
        chat_history.append({"role": "assistant", "content": f"**Bot 1 ({persona1}):** {bot1_content}"})
        yield chat_history

        message_updater(messages=messages, role='assistant', system_prompt=system_message2)
        response = anthropic2.chat.completions.create(model=ANTHROPIC_MODEL, messages=messages)
        bot2_content = response.choices[0].message.content
        print(f"Bot 2 ({persona2}): {bot2_content}")
        messages.append({'role': 'assistant', 'content': bot2_content})
        chat_history.append({"role": "user", "content": f"**Bot 2 ({persona2}):** {bot2_content}"})
        yield chat_history


persona_choices = list(PERSONAS.keys())

with gr.Blocks() as demo:
    gr.Markdown("## LLM Fireside Chat")
    chatbot = gr.Chatbot(type="messages", height=500, label="Bot 1 (left) vs Bot 2 (right)")
    with gr.Row():
        persona1_dd = gr.Dropdown(choices=persona_choices, value=persona_choices[0], label="Bot 1 Persona", scale=2)
        persona2_dd = gr.Dropdown(choices=persona_choices, value=persona_choices[1], label="Bot 2 Persona", scale=2)
    with gr.Row():
        topic_box = gr.Textbox(value="lets discuss about Marketcast", label="Topic", scale=4)
        start_btn = gr.Button("Start Chat", variant="primary", scale=1)

    start_btn.click(fn=start_chat, inputs=[topic_box, persona1_dd, persona2_dd], outputs=chatbot)

demo.launch()